# Seeing with Transformers: ViT, CLIP and Vision-Language Models

The full code of [Seeing with Transformers: ViT, CLIP and Vision-Language Models](https://residual-stream.lvtan71.workers.dev/posts/vision-transformers-and-multimodal/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/vision-transformers-and-multimodal) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vision-transformers-and-multimodal.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 0 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q numpy "torch==2.14.*" "transformers==5.17.*" pillow huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "vision-transformers-and-multimodal"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/vision-transformers-and-multimodal


In [2]:
!tar xzf blog-11f17f9.tar.gz

In [3]:
!tar xzf clip-diagram-renders.tar.gz

## `patches.py`

A 224x224 image as ViT-B/16 patch tokens: patch count, patch-embedding parameters, sequence length at several resolutions, and the KV cache one image costs in LLaVA-1.5.

In [4]:
!{sys.executable} patches.py

224x224 image, 16x16 patches: 14 x 14 = 196 patches, each 768 numbers
patch embedding: a 768 x 768 matrix + bias = 590,592 parameters
sequence for the Transformer: 196 patch tokens + 1 [CLS] = 197 positions, attention over 38,809 pairs
  224x224 with 14x14 patches: 256 tokens
  336x336 with 14x14 patches: 576 tokens
  448x448 with 14x14 patches: 1,024 tokens
  1024x1024 with 16x16 patches: 4,096 tokens

LLaVA-1.5 (7B language model): 512 KiB of KV cache per token
  one image, 576 tokens: 288 MiB; a 20-image conversation: 5.6 GiB
  as text, 576 tokens is about 432 English words


The post printed:

```text
224x224 image, 16x16 patches: 14 x 14 = 196 patches, each 768 numbers
patch embedding: a 768 x 768 matrix + bias = 590,592 parameters
sequence for the Transformer: 196 patch tokens + 1 [CLS] = 197 positions, attention over 38,809 pairs
  224x224 with 14x14 patches: 256 tokens
  336x336 with 14x14 patches: 576 tokens
  448x448 with 14x14 patches: 1,024 tokens
  1024x1024 with 16x16 patches: 4,096 tokens

LLaVA-1.5 (7B language model): 512 KiB of KV cache per token
  one image, 576 tokens: 288 MiB; a 20-image conversation: 5.6 GiB
  as text, 576 tokens is about 432 English words
```

## `clip_loss.py`

CLIP's symmetric InfoNCE loss and SigLIP's pairwise sigmoid loss on a batch of three image-caption pairs.

In [5]:
!{sys.executable} clip_loss.py

logits = similarities / 0.07:
 [[12.86  2.86  1.43]
 [ 4.29 11.43  5.71]
 [ 0.71  5.   10.  ]]
image -> caption losses: [0.0001 0.0041 0.0068]  caption -> image losses: [0.0002 0.0018 0.0139]
CLIP loss (symmetric InfoNCE) = mean of both directions = 0.0045

SigLIP per-pair losses:
 [[1.3133e+00 3.0000e-04 1.0000e-04]
 [9.0000e-04 2.1269e+00 2.5000e-03]
 [1.0000e-04 1.5000e-03 3.0486e+00]]
SigLIP loss = sum over all 9 pairs / 3 images = 2.1647


The post printed:

```text
logits = similarities / 0.07:
 [[12.86  2.86  1.43]
 [ 4.29 11.43  5.71]
 [ 0.71  5.   10.  ]]
image -> caption losses: [0.0001 0.0041 0.0068]  caption -> image losses: [0.0002 0.0018 0.0139]
CLIP loss (symmetric InfoNCE) = mean of both directions = 0.0045

SigLIP per-pair losses:
 [[1.3133e+00 3.0000e-04 1.0000e-04]
 [9.0000e-04 2.1269e+00 2.5000e-03]
 [1.0000e-04 1.5000e-03 3.0486e+00]]
SigLIP loss = sum over all 9 pairs / 3 images = 2.1647
```

## `clip_diagrams.py`

CLIP ViT-B/32 matching 19 diagrams to their posts' alt texts and to their file names, as drawn and with labels hidden, in both directions.

In [6]:
!{sys.executable} clip_diagrams.py

19 diagrams; chance of the right match at random: 1 in 19 (5%)

diagrams             caption                   caption -> diagram  diagram -> caption  (right one first)


as drawn             the post's alt text                 5 of 19            4 of 19


as drawn             the file name as words              4 of 19            3 of 19


with labels hidden   the post's alt text                 1 of 19            2 of 19


with labels hidden   the file name as words              1 of 19            2 of 19
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (85 > 77). Running this sequence through the model will result in indexing errors

alt texts: median 40 tokens, 3 of 19 cut at 77


The post printed:

```text
19 diagrams; chance of the right match at random: 1 in 19 (5%)

diagrams             caption                   caption -> diagram  diagram -> caption  (right one first)
as drawn             the post's alt text                 5 of 19            4 of 19
as drawn             the file name as words              4 of 19            3 of 19
with labels hidden   the post's alt text                 1 of 19            2 of 19
with labels hidden   the file name as words              1 of 19            2 of 19

alt texts: median 40 tokens, 3 of 19 cut at 77
```